# DQMBot — Batch Image Query Driver

In [3]:
import sys
from pathlib import Path
import pandas as pd
import re

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    list_models, get_knowledge_map, query,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
)

print('owui_client loaded OK')

owui_client loaded OK


In [4]:
# ── Discover available models and knowledge collections ───────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

print()
print('=== Knowledge collections ===')
kb_map = get_knowledge_map()
# for name, kid in kb_map.items():
#     print(f'  {name:30s}  {kid}')

=== Models ===
  vllm.gpt-oss:120b
  litellm-ow.google/gemma4-31b
  litellm-ow.qwen/qwen3.6
  nonfree.azure/auto
  nonfree.azure/gpt-5-nano
  nonfree.azure/gpt-5.4
  nonfree.azure/claude-opus-4-8
  835
  acnet-documentation
  acnet-documentation-coder
  dqm-chatbot
  dqmtest
  gemma3:latest
  qwen2.5vl:latest
  qwen3-vl:latest
  erlang-otp
  litellm-ow.qwen/qwen3-coder-next
  qwen3-vl:32b
  srf-data-test
  stockroom
  aeolus
  qwen2.5:7b
  stockroom-clone-hayden
  stockroom-clone-hayden-medium
  litellm-ow.qwen/qwen35-9b
  qwenqwen35-9b-no-thinking
  policies
  ad-kubernetes
  qwen3-vl-longctx32b
  qwen3-vl-longctx

=== Knowledge collections ===


In [5]:
# ── Run → event type mapping ──────────────────────────────────────────────────
EVENT_TYPE_MAP = {
    398185: 'collisions',
    398186: 'cosmics',
    398187: 'circulating',
    398188: 'collisions',
    398189: 'collisions',
    398191: 'collisions',
    398194: 'cosmics',
    398199: 'cosmics',
}

def get_prompt(image_path: Path) -> str:
    match = re.search(r'run(\d+)', image_path.stem)
    run   = int(match.group(1)) if match else None
    event = EVENT_TYPE_MAP.get(run, 'unknown')
    return f'The plot is from "{event}" event.'

In [6]:
# ── Configuration ─────────────────────────────────────────────────────────────
# IMAGE_ROOT  = Path('images')
# RUN_ID = 'goodtest'
# IMAGE_ROOT  = Path('bad_images')
# RUN_ID = 'badtest'
IMAGE_ROOT  = Path('images/Ecal_03_Occupancy')
RUN_ID = 'ECAL_03'
OUTPUT_ROOT = Path('results')

# Give the name of the reference image to the prompt
REF_IMAGE = "00 - CaloLayer1 ECAL occupancy"
REF_PDF   = "CMS_DQMShiftL1T.pdf"

BEST_MODEL = ['litellm-ow.google/gemma4-31b']


MODELS = [
 'qwen2.5vl:latest',            #7b    
 'qwen3-vl-longctx', #8b
 'qwen3-vl-longctx32b', #32b 
 'gemma3:latest',               #4b
 'litellm-ow.google/gemma4-31b',#31b
]

COLLECTIONS = [
    kb_map['DQM shift rules'],
]

SYSTEM_PROMPT = (
"""\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad. If the input is 00 - CaloLayer1 ECAL occupancy plot, 
ignore the vertical white space at iEta = 0.

In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction
 - Decide if the plot is good or bad\
"""
)

SYSTEM_PROMPT_WITH_REF = (
 f"""\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad. The reference plot is "{REF_IMAGE}" in the knowledge. 
If the input is 00 - CaloLayer1 ECAL occupancy plot, ignore the vertical white space at iEta = 0.
 
In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction and the reference plot "{REF_IMAGE}"
 - Decide if the plot is good or bad\
"""
)


SYSTEM_PROMPT_WITH_REFS = (
f"""\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad. The reference plot is "{REF_IMAGE}" in the knowledge.
Detailed shift instructions can be found in "{REF_PDF}" in the knowledge. If the input is 00 - CaloLayer1 ECAL occupancy plot, 
ignore the vertical white space at iEta = 0.

In your output, make 4 sections:
 - Quote the relevant section of instructions from "{REF_PDF}" for the input plot
 - Describe the input plot
 - Compare input plot to the instruction and the reference plot "{REF_IMAGE}"
 - Decide if the plot is good or bad\
"""
)

PROMPT = (
    """
    """
)

DELAY = 1.5
# ──────────────────────────────────────────────────────────────────────────────

In [7]:
# # ── Sanity check: show what will be processed and where it will land ──────────

# IMAGE_ROOT  = Path('bad_images')
# OUTPUT_ROOT = Path('results')
# # Set a string to preserve previous runs alongside this one.
# # Leave as None to overwrite.
# RUN_ID = 'badtest'


# pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
# plot_names = sorted(set(p for p, _ in pairs))

# print(f'Plots found  : {len(plot_names)}')
# for pn in plot_names:
#     imgs = [img for p, img in pairs if p == pn]
#     print(f'  {pn}/  ({len(imgs)} images)')
#     for img in imgs:
#         print(f'    {img.name}')

# print(f'\nModels       : {len(MODELS)}')
# for m in MODELS:
#     print(f'  {m}')

# print(f'\nRun ID       : {RUN_ID or "(none — overwrite mode)"}')
# print(f'Total queries: {len(pairs) * len(MODELS)}')

# print('\nExample output paths:')
# for model in MODELS:
#     plot_name, img = pairs[0]
#     d = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
#     f = resolve_output_file(d, img, model)
#     print(f'  {f}')

In [8]:
# # ── Warm-model test: compare load_latency on first vs second query ────────
# # Run the same prompt twice back-to-back on the same model.
# # Run 1 pays the cold-start cost (model loading from disk).
# # Run 2 should show load_latency_s ≈ 0 if the model stayed warm in memory.

# pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))

# for i in range(0, len(MODELS)):
#     TEST_MODEL = MODELS[i]   # change to any model you want to test
    
#     if pairs:
#         plot_name, img = pairs[0]
#         print(f"Model : {TEST_MODEL}")
#         print(f"Image : {img.name}")
#         print()
    
#         for run_num in [1, 2, 3]:
#             result = query(
#                 PROMPT,
#                 model=TEST_MODEL,
#                 system=SYSTEM_PROMPT,
#                 image_path=img,
#                 collection_ids=COLLECTIONS,
#             )
#             label = '<-- cold start (model loading)' if run_num == 1 else '<-- should be ~0 if model is warm'
#             print(f"Run {run_num}:")
#             print(f"  load_latency_s       : {result['load_latency_s']}s  {label}")
#             print(f"  generation_latency_s : {result['generation_latency_s']}s")
#             print(f"  total_latency_s      : {result['latency_s']}s")
#             if result['error']:
#                 print(f"  ERROR: {result['error']}")
#             print()

In [9]:
# # ── Smoke test: one image, first model ───────────────────────────────────────
# pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
# if pairs:
#     plot_name, img = pairs[1]
#     test = query(
#         PROMPT,
#         model=MODELS[4],
#         system=SYSTEM_PROMPT_WITH_REF,
#         image_path=img,
#         collection_ids=COLLECTIONS,
#         # think=False,
#         # no_think=True,
#         # num_ctx=16384,
#     )
#     print(f"Plot    : {plot_name}")
#     print(f"Model   : {test['model']}")
#     print(f"Model used   : {test['model_used']}")
#     print(f"Image   : {test['image']}")
#     print(f"Load latency : {test['load_latency_s']}s")
#     print(f"Generation latency : {test['generation_latency_s']}s")
#     print(f"Total latency : {test['latency_s']}s")
#     print(f"Error   : {test['error']}")
#     # print()
#     print(test['response'])
#     print("------------------------------")

In [10]:
# pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
# if pairs:
#     plot_name, img = pairs[1]
#     test = query(
#         PROMPT,
#         model=MODELS[2],
#         system=SYSTEM_PROMPT_WITH_REF,
#         image_path=img,
#         collection_ids=COLLECTIONS,
#         # think=False,
#         # no_think=True,
#         # num_ctx=16384,
#     )
#     print(f"Plot    : {plot_name}")
#     print(f"Model   : {test['model']}")
#     print(f"Model used   : {test['model_used']}")
#     print(f"Image   : {test['image']}")
#     print(f"Load latency : {test['load_latency_s']}s")
#     print(f"Generation latency : {test['generation_latency_s']}s")
#     print(f"Total latency : {test['latency_s']}s")
#     print(f"Error   : {test['error']}")
#     # print()
#     print(test['response'])
#     print("------------------------------")

### Run best model on ECAL_03

In [11]:
all_results = {}
pairs      = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
prompt_map = {image_path: get_prompt(image_path) for plot_name, image_path in pairs}

for label, sys_prompt in [
    ("no_ref",    SYSTEM_PROMPT),
]:
    run_label = f"{RUN_ID}_{label}" if RUN_ID else label
    print(f"\n{'='*60}")
    print(f"Running: {run_label}")
    print(f"{'='*60}")

    results = batch_query_images(
        PROMPT,
        image_root=IMAGE_ROOT,
        models=BEST_MODEL,
        output_root=OUTPUT_ROOT,
        run_id=run_label,
        system=sys_prompt,
        collection_ids=COLLECTIONS,
        delay=DELAY,
        verbose=False,
        prompt_map=prompt_map,
    )
    all_results[label] = results
    print(f'Done. {len(results)} queries completed.')

print(f'\nAll batches done. Results in:')
for label in all_results:
    run_label = f"{RUN_ID}_{label}" if RUN_ID else label
    print(f'  {OUTPUT_ROOT}/{run_label}/')


Running: ECAL_03_no_ref
Done. 4 queries completed.

All batches done. Results in:
  results/ECAL_03_no_ref/


### Run on all images

In [9]:
# Run the entire batch with and without giving the reference plot name 
# and giving both ref_plot name and ref_PDF in the prompt.
all_results = {}
pairs      = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
prompt_map = {image_path: get_prompt(image_path) for plot_name, image_path in pairs}

for label, sys_prompt in [
    ("no_ref",    SYSTEM_PROMPT),
    ("with_ref",    SYSTEM_PROMPT_WITH_REF),
    ("with_refs", SYSTEM_PROMPT_WITH_REFS),
]:
    run_label = f"{RUN_ID}_{label}" if RUN_ID else label
    print(f"\n{'='*60}")
    print(f"Running: {run_label}")
    print(f"{'='*60}")

    results = batch_query_images(
        PROMPT,
        image_root=IMAGE_ROOT,
        models=MODELS,
        output_root=OUTPUT_ROOT,
        run_id=run_label,
        system=sys_prompt,
        collection_ids=COLLECTIONS,
        delay=DELAY,
        verbose=False,
        prompt_map=prompt_map,
    )
    all_results[label] = results
    print(f'Done. {len(results)} queries completed.')

print(f'\nAll batches done. Results in:')
for label in all_results:
    run_label = f"{RUN_ID}_{label}" if RUN_ID else label
    print(f'  {OUTPUT_ROOT}/{run_label}/')


Running: badtest_no_ref
Done. 15 queries completed.

Running: badtest_with_ref
Done. 15 queries completed.

Running: badtest_with_refs
Done. 15 queries completed.

All batches done. Results in:
  results/badtest_no_ref/
  results/badtest_with_ref/
  results/badtest_with_refs/


In [6]:
# # ── Full batch ────────────────────────────────────────────────────────────────
# OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# results = batch_query_images(
#     PROMPT,
#     image_root=IMAGE_ROOT,
#     models=MODELS,
#     output_root=OUTPUT_ROOT,
#     run_id=RUN_ID,
#     system=SYSTEM_PROMPT,
#     collection_ids=COLLECTIONS,
#     delay=DELAY,
#     verbose=True,
# )

# print(f'\nDone. {len(results)} queries completed.')

[1/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ECALoccupancy_run398185.png ... 31.98s → results/baseline/ECALoccupancy/ECALoccupancy_run398185_qwen2.5vl_latest.txt
[2/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ecalOccRecdEtWgt_run398185.png ... 6.09s → results/baseline/ECALoccupancy/ecalOccRecdEtWgt_run398185_qwen2.5vl_latest.txt
[3/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ecalOccRecdEtWgt_run398186.png ... 5.66s → results/baseline/ECALoccupancy/ecalOccRecdEtWgt_run398186_qwen2.5vl_latest.txt
[4/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ecalOccRecdEtWgt_run398187.png ... 8.69s → results/baseline/ECALoccupancy/ecalOccRecdEtWgt_run398187_qwen2.5vl_latest.txt
[5/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ecalOccRecdEtWgt_run398188.png ... 6.51s → results/baseline/ECALoccupancy/ecalOccRecdEtWgt_run398188_qwen2.5vl_latest.txt
[6/18] model=qwen2.5vl:latest  plot=ECALoccupancy  image=ecalOccRecdEtWgt_run398189.png ... 5.1s → results/bas

In [ ]:
# ── Retry errors ──────────────────────────────────────────────────────────────
# Re-runs only queries that errored in the current session's `results` list.
# Merges successful retries back into `results` in-place.

failed = [
    (r['model_used'], r['image'])
    for r in results if r['error'] is not None
]

if not failed:
    print('No errors in results — nothing to retry.')
else:
    print(f'Retrying {len(failed)} failed quer{"y" if len(failed)==1 else "ies"}...')
    retry_results = []

    for i, (model, image_str) in enumerate(failed):
        image_path = Path(image_str)
        plot_name  = image_path.parent.name
        print(f'  [{i+1}/{len(failed)}] model={model}  image={image_path.name} ...', end=' ', flush=True)

        result = query(
            PROMPT,
            model=model,
            system=SYSTEM_PROMPT,
            image_path=image_path,
            collection_ids=COLLECTIONS,
        )
        result['plot_name'] = plot_name
        retry_results.append(result)

        out_dir  = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = resolve_output_file(out_dir, image_path, model)
        with open(out_file, 'w') as f:
            f.write(f"Model:    {result['model']}\n")
            f.write(f"Plot:     {plot_name}\n")
            f.write(f"Image:    {result['image']}\n")
            f.write(f"Run ID:   {RUN_ID or '(overwrite)'}\n")
            f.write(f"Latency:  {result['latency_s']}s\n")
            f.write(f"Prompt:   {result['prompt']}\n")
            f.write('-' * 60 + '\n')
            if result['error']:
                f.write(f"ERROR: {result['error']}\n")
            else:
                f.write(result['response'] + '\n')

        status = 'ERROR' if result['error'] else f"{result['latency_s']}s → {out_file}"
        print(status)
        time.sleep(DELAY)

    # Merge back into results
    retry_index = {(r['model_used'], r['image']): r for r in retry_results}
    results = [
        retry_index.get((r['model_used'], r['image']), r)
        for r in results
    ]

    still_failing = sum(1 for r in retry_results if r['error'])
    print(f'\nDone. {len(retry_results) - still_failing}/{len(retry_results)} recovered.')
    if still_failing:
        print('Still failing:')
        for r in retry_results:
            if r['error']:
                print(f"  {r['model_used']}  {Path(r['image']).name}  → {r['error']}")

In [11]:
# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model_used', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model_used'])['latency_s']
      .agg(['count', 'mean', 'min', 'max'])
      .round(2)
      .rename(columns={'count': 'n', 'mean': 'avg_s', 'min': 'min_s', 'max': 'max_s'})
)

All queries succeeded.



n  avg_s  min_s  max_s
plot_name        model_used                               
ecalOccRecdEtWgt gemma3:latest      1  13.91  13.91  13.91
                 google/gemma4-31b  1  51.39  51.39  51.39
                 qwen/qwen3.6       1  45.27  45.27  45.27
                 qwen2.5vl:32b      1  38.58  38.58  38.58
                 qwen2.5vl:latest   1  16.89  16.89  16.89
                 qwen3-vl:latest    1  98.53  98.53  98.53

In [12]:
# ── Save CSV next to the run's output folder ──────────────────────────────────
if RUN_ID:
    csv_path = OUTPUT_ROOT / RUN_ID / f'summary_{RUN_ID}.csv'
else:
    csv_path = OUTPUT_ROOT / 'summary.csv'

df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Show result tree
print()
root = OUTPUT_ROOT / RUN_ID if RUN_ID else OUTPUT_ROOT
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

Saved: results/baseline/summary_baseline.csv

  ecalOccRecdEtWgt/  (6 files)
    ecalOccRecdEtWgt_run398185_gemma3_latest.txt
    ecalOccRecdEtWgt_run398185_litellm-ow.google_gemma4-31b.txt
    ecalOccRecdEtWgt_run398185_litellm-ow.qwen_qwen3.6.txt
    ecalOccRecdEtWgt_run398185_qwen2.5vl_32b.txt
    ecalOccRecdEtWgt_run398185_qwen2.5vl_latest.txt
    ecalOccRecdEtWgt_run398185_qwen3-vl_latest.txt
  summary_baseline.csv


In [13]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = plot_names[0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model_used']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    print(row['error'] and f"ERROR: {row['error']}" or row['response'])
    print()

Model   : qwen2.5vl:latest
Latency : 16.89s

### Instructions for the Input Plot:
The plot is titled "ECal TP ET-weighted Occupancy at Layer1". It shows the occupancy of ECal (Electromagnetic Calorimeter) at Layer1, weighted by the energy transverse (ET) of the deposited energy. The x-axis represents the iEta (eta index), and the y-axis represents the iPhi (phi index). The color bar on the right indicates the number of entries, with a range from 0 to 80,000.

### Description of the Input Plot:
The plot is a heatmap representing the occupancy of ECal at Layer1, with the energy transverse (ET) weight. The x-axis is labeled as iEta, ranging from approximately -25 to 25, and the y-axis is labeled as iPhi, ranging from 0 to 70. The color intensity varies from dark blue (low occupancy) to red (high occupancy), with a legend on the right indicating the number of entries. The plot shows a clear pattern where the occupancy is higher in certain regions, particularly in the central part of the pl